## RAG

In [ ]:
print("git push testt")

In [1]:
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
import os

C:\Users\lenovo\AppData\Local\Temp\ipykernel_7368\2053975123.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import DirectoryLoader, TextLoader
c:\Users\lenovo\New_Document\Coding\Project AI Builders\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
loader = DirectoryLoader(
    path='sms_scam_policies/', 
    glob="**/*.txt",               
    loader_cls=TextLoader,         
    loader_kwargs={"encoding": "utf-8"}
)

documents = loader.load()

NameError: name 'DirectoryLoader' is not defined

In [3]:
for i, doc in enumerate(documents, 1):
    print(doc.metadata['source']) 
    print(doc.page_content[:100])  


sms_scam_policies\Doc_01.txt
นโยบายอ้างอิง: ตามประกาศธนาคารแห่งประเทศไทย (ธปท.) ตั้งแต่วันที่ 24 มี.ค. 2566 สถาบันการเงินและหน่วย
sms_scam_policies\Doc_02.txt
นโยบายอ้างอิง: นโยบายความปลอดภัยของกลุ่มธุรกิจประกันภัย
เงื่อนไขการตรวจจับแสกม: บริษัทจะไม่มีการส่ง 
sms_scam_policies\Doc_03.txt
นโยบายอ้างอิง: มาตรการปราบปรามอาชญากรรมทางเทคโนโลยีจาก กสทช. และนโยบายความปลอดภัยของผู้ให้บริการ
เงื
sms_scam_policies\Doc_04.txt
นโยบายอ้างอิง: นโยบายการสื่อสารมาตรฐานขององค์กรธุรกิจ แบรนด์ และแพลตฟอร์ม E-Commerce
เงื่อนไขการตรวจ
sms_scam_policies\Doc_05.txt
นโยบายอ้างอิง (กรณีพิเศษ): อิงตาม พ.ร.บ. ว่าด้วยการกระทำความผิดเกี่ยวกับคอมพิวเตอร์ และ พ.ร.บ. การพน


In [4]:
# Chunking

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,              
    chunk_overlap=100,          
    separators=["\n\n", "\n", "。", "、", " ", ""]
)

chunks = text_splitter.split_documents(documents)

print(f"  - จำนวนเอกสารเดิม: {len(documents)}")
print(f"  - จำนวน chunks : {len(chunks)}")
print(chunks[0].page_content[:200])

  - จำนวนเอกสารเดิม: 5
  - จำนวน chunks : 9
นโยบายอ้างอิง: ตามประกาศธนาคารแห่งประเทศไทย (ธปท.) ตั้งแต่วันที่ 24 มี.ค. 2566 สถาบันการเงินและหน่วยงานของรัฐ "ยกเลิกการส่ง SMS แนบลิงก์ทุกกรณี"
เงื่อนไขการตรวจจับแสกม: หากมีการส่ง SMS ที่แนบลิงก์มายั


In [5]:
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True}
)

print("Embedding Model loaded")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3158.66it/s]


Embedding Model loaded


In [6]:
# Vectore Store
persist_directory = "./chroma_db"

if os.path.exists(persist_directory):
    vectorstore = Chroma(
        persist_directory=persist_directory,
        embedding_function=embedding_model
    )
    
    print(f"โหลด: {vectorstore._collection.count()} vectors")
else:
    print("สร้าง Vector Store ใหม่")
    # สร้าง vector store จาก chunks
    vectorstore = Chroma.from_documents(
        documents=chunks,                   
        embedding=embedding_model,          
        persist_directory=persist_directory,
        collection_name="scam_policies"    
    )
    
    vectorstore.persist()
    print(f"save: {len(chunks)} vectors")

สร้าง Vector Store ใหม่
save: 9 vectors


C:\Users\lenovo\AppData\Local\Temp\ipykernel_7368\891177838.py:21: LangChainDeprecationWarning: Since Chroma 0.4.x the manual persistence method is no longer supported as docs are automatically persisted.
  vectorstore.persist()


In [ ]:
retriever = vectorstore.as_retriever(
    search_type="similarity",           
    search_kwargs={"k": 3}    # ดึงมา 3 ข้อความที่คล้ายมากที่สุด
)

In [ ]:
#ทดสอบ Retriever

test_query = "คุณได้รับเงิน 50,000 บาทจากการถูกหวย โปรดคลิกลิงก์นี้เพื่อยืนยันตัวตนของคุณ"
retrieved_docs = retriever.invoke(test_query)

# แสดงผลลัพธ์
print(f"\nพบข้อมูลที่เกี่ยวข้อง: {len(retrieved_docs)} รายการ\n")

for i, doc in enumerate(retrieved_docs, 1):
    print(f"ผลลัพธ์ที่ {i}:")
    print(f"ที่มา: {doc.metadata['source']}")
    print(f"เนื้อหา: {doc.page_content[:300]}...")
    print("-" * 50)


พบข้อมูลที่เกี่ยวข้อง: 3 รายการ

ผลลัพธ์ที่ 1:
ที่มา: sms_scam_policies\Doc_05.txt
เนื้อหา: รายชื่อหมวดหมู่ (4 รายการ): Lawyer Scam, Online Gambling, Unknown/General Scam, slot...
--------------------------------------------------
ผลลัพธ์ที่ 2:
ที่มา: sms_scam_policies\Doc_05.txt
เนื้อหา: นโยบายอ้างอิง (กรณีพิเศษ): อิงตาม พ.ร.บ. ว่าด้วยการกระทำความผิดเกี่ยวกับคอมพิวเตอร์ และ พ.ร.บ. การพนัน
เงื่อนไขการตรวจจับแสกม: เอนทิตีกลุ่มนี้ไม่มี "นโยบายบริษัท" ที่ถูกกฎหมายรองรับ เนื่องจากกิจกรรมเหล่านี้ผิดกฎหมายหรือเป็นกระบวนการหลอกลวงตั้งแต่ต้นทาง ดังนั้น SMS ทุกรูปแบบที่เชิญชวนเล่นพนัน (Slot),...
--------------------------------------------------
ผลลัพธ์ที่ 3:
ที่มา: sms_scam_policies\Doc_04.txt
เนื้อหา: รายชื่อบริษัท/หมวดหมู่ (51 รายการ): 7-Eleven, Beauty Clinic, Big C, Billing / Rewards, Boots, Central, Central / The 1, Disney+, Facebook, GMM Grammy, Grab, HomePro, Hospital/Foundation, IKEA, Job Recruitment, KMUTT, Lancome, Lazada, lazada, Lenovo, Lotus's, Lucky Draw/Giveaway, Major Cineplex,